## Function kernel

The function kernel facilitates calling Python functions, MCP tools and REST servers.

All function inputs and outputs are defined as Pydantic models to simplify validation. However, the function kernel does its best to convert the input and output of functions that use plain data types to.

In [1]:
import datetime
from kavalai.functionkernel import FunctionKernel, pythontool

@pythontool
def get_time() -> str:
    """Returns current time in UTC"""
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

# Instantiate a function kernel and register the get_time tool.
kernel = FunctionKernel()
kernel.register_python_tool("get_time", get_time)

# Input model with no fields, output model plain string.
# By default "result" field is defined for functions that are not
# Pydantic models.
print(kernel.get_input_model("python://get_time").model_fields)
print(kernel.get_output_model("python://get_time").model_fields)
await kernel.call_tool("python://get_time")

/home/timo/projects/kaval.ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{}
{'result': FieldInfo(annotation=str, required=True)}


get_time_output(result='2026-06-10T22:17:53.215274+00:00')

## v2 Agent

`kavalai.agents.v2.agent.Agent` is a lightweight multi-step reasoning loop that:

1. Renders a system prompt from a Jinja2 template with the task, available tools, and step history.
2. Calls the LLM to get a `StepOutput` — a list of tool calls **plus** an optional final answer.
3. Executes tool calls in parallel through the `FunctionKernel` and feeds results back into the next step.
4. Stops when the model returns an answer without further tool calls or `max_steps` is reached.

Any v2 LLM client (`OpenAIClient`, `GeminiClient`, `OllamaClient`) can be plugged in.

In [2]:
# Load environment variables from the project .env file
from dotenv import load_dotenv
load_dotenv()

from kavalai.agents.v2.agent import Agent
from kavalai.agents.run_context import RunContext
from kavalai.agents.workflow_model import RestServer, McpServer
from kavalai.functionkernel import FunctionKernel, pythontool
from kavalai.llm_clients.v2.gemini_client import GeminiClient
from kavalai.llm_clients.v2.openai_client import OpenAIClient

### Example 1 — Business Info Agent (Python tools)

Port of `examples/business_info_agent.py` to the v2 Agent API.

The agent uses two Python tools registered in the `FunctionKernel`:
- `python://websearch.serper_web_search` — Google Search via Serper API
- `python://webtools.serper_scrape_url` — web scraping via Serper Scrape API

It reasons over them to fill in a structured `BusinessInfo` Pydantic model.

In [3]:
from typing import Optional
from pydantic import BaseModel, Field, ConfigDict


class BusinessInfo(BaseModel):
    """Structured information about a business found online."""

    model_config = ConfigDict(extra="forbid")

    name: str = Field(description="The legal or trading name of the business.")
    address: Optional[str] = Field(None, description="The physical address.")
    website: Optional[str] = Field(None, description="The official website URL.")
    phone: Optional[str] = Field(None, description="Contact phone number.")
    owners: Optional[str] = Field(None, description="The owners of the business.")
    description: str = Field(description="Brief description of what the business does.")
    industry: Optional[str] = Field(None, description="The industry it operates in.")

In [4]:
import json
from kavalai.tools.websearch.serper import serper_web_search
from kavalai.tools.webtools.serper_scraper import serper_scrape_url

# Build kernel with the two serper tools
kernel = FunctionKernel()
kernel.register_python_tool("websearch.serper_web_search", serper_web_search)
kernel.register_python_tool("webtools.serper_scrape_url", serper_scrape_url)

# v2 LLM client — swap to OpenAIClient("gpt-4.1-mini") if preferred
llm_client = GeminiClient("gemini-2.5-flash")

agent = Agent(llm_client=llm_client, kernel=kernel)

result = await agent.prompt(
    prompt=(
        "Find information about the company 'Kaval AI' online. "
        "Search for it first, then scrape the most relevant source to fill in all fields."
    ),
    response_model=BusinessInfo,
    max_steps=5,
)

print(json.dumps(result.model_dump(), indent=2))

2026-06-11 01:17:53.576 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 0/5
2026-06-11 01:17:55.039 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 2011 tokens, 1.46s
2026-06-11 01:17:55.041 | INFO     | kavalai.agents.v2.agent:_call_tool:241 - Calling tool python://websearch.serper_web_search with {'query': 'Kaval AI'}
2026-06-11 01:17:55.813 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 1/5
2026-06-11 01:17:57.604 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 2970 tokens, 1.79s
2026-06-11 01:17:57.606 | INFO     | kavalai.agents.v2.agent:_call_tool:241 - Calling tool python://webtools.serper_scrape_url with {'url': 'https://kaval.ai/'}
2026-06-11 01:18:01.862 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 2/5
2026-06-11 01:18:03.584 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini

{
  "name": "Kaval.AI",
  "address": null,
  "website": "https://kaval.ai/",
  "phone": null,
  "owners": null,
  "description": "AI agent consultancy and development services company.",
  "industry": "AI Consultancy"
}


### Example 2 — REST tools

REST servers are registered with a base URL. Individual endpoints are registered as tools with their HTTP method, input/output JSON schemas, and a description.

This example uses the free [Open-Meteo](https://open-meteo.com/) weather API — no API key required.

In [5]:
kernel_rest = FunctionKernel()

# Register the Open-Meteo REST server
kernel_rest.register_rest_server(
    RestServer(name="weather", url="https://api.open-meteo.com/v1")
)

# Register the /forecast endpoint as a tool
kernel_rest.register_rest_tool(
    server_name="weather",
    tool_name="forecast",
    method="GET",
    input_schema={
        "type": "object",
        "properties": {
            "latitude": {"type": "number", "description": "Latitude of the location."},
            "longitude": {"type": "number", "description": "Longitude of the location."},
            "current": {
                "type": "string",
                "description": "Comma-separated list of current weather variables, e.g. temperature_2m,wind_speed_10m",
            },
        },
        "required": ["latitude", "longitude"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"},
            "current": {"type": "object"},
        },
    },
    description="Get current weather data for a GPS location from Open-Meteo.",
)

llm_client_rest = GeminiClient("gemini-2.5-flash")
agent_rest = Agent(llm_client=llm_client_rest, kernel=kernel_rest)

result_rest = await agent_rest.prompt(
    prompt="What is the current temperature and wind speed in Tallinn, Estonia?",
    max_steps=3,
)

print(result_rest)

2026-06-11 01:18:03.879 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 0/3
2026-06-11 01:18:05.673 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 722 tokens, 1.79s
2026-06-11 01:18:05.675 | INFO     | kavalai.agents.v2.agent:_call_tool:241 - Calling tool rest://weather.forecast [GET] with {'latitude': 59.437, 'longitude': 24.7536, 'current': 'temperature_2m,wind_speed_10m'}
2026-06-11 01:18:05.749 | INFO     | kavalai.functionkernel:_call_rest_tool:381 - Calling GET https://api.open-meteo.com/v1/forecast
2026-06-11 01:18:05.927 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 1/3
2026-06-11 01:18:33.505 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 7581 tokens, 27.58s


The current temperature in Tallinn, Estonia is 15.4






















































































































































































































































































































































































































































































































































































































































































































































































































































































































































































### Example 3 — MCP tools (stdio server)

MCP servers are registered with a shell command. The `FunctionKernel` starts the process, speaks the MCP JSON-RPC protocol over stdio, discovers available tools, and routes calls to them.

Here we spin up an in-process `FastMCP` server that exposes two arithmetic tools.

In [6]:
import sys
import textwrap

# Write a minimal FastMCP server to a temp file so the kernel can start it
mcp_server_path = "/tmp/kavalai_demo_mcp_server.py"
with open(mcp_server_path, "w") as f:
    f.write(textwrap.dedent("""
        from mcp.server.fastmcp import FastMCP

        mcp = FastMCP("demo-math")

        @mcp.tool()
        def add(a: float, b: float) -> float:
            \"\"\"Add two numbers.\"\"\"
            return a + b

        @mcp.tool()
        def multiply(a: float, b: float) -> float:
            \"\"\"Multiply two numbers.\"\"\"
            return a * b

        if __name__ == "__main__":
            mcp.run()
    """))

kernel_mcp = FunctionKernel()
kernel_mcp.register_mcp_server(
    McpServer(
        name="math",
        command=sys.executable,
        args=[mcp_server_path],
    )
)

llm_client_mcp = GeminiClient("gemini-2.5-flash")
agent_mcp = Agent(llm_client=llm_client_mcp, kernel=kernel_mcp)

result_mcp = await agent_mcp.prompt(
    prompt="What is (42 + 8) multiplied by 7?",
    max_steps=3,
)

print(result_mcp)
await kernel_mcp.close()

2026-06-11 01:18:33.757 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 0/3
2026-06-11 01:18:34.799 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 330 tokens, 1.04s


350
